# ជំពូក ៣៖ ការសរសេរកូដ Attention Mechanisms

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រៀមបរិស្ថាន
- ## ១. Attention សាមញ្ញបំផុត
- ## ២. Attention scores និង context vectors
- ## ៣. Trainable Q/K/V weights
- ## ៤. ក្លាស SelfAttention_v2
- ## ៥. Causal mask
- ## ៦. Dropout និង batch
- ## ៧. ក្លាស CausalAttention
- ## ៨. Multi-head attention តាម wrapper
- ## ៩. ក្លាស MultiHeadAttention បង្រួម

## ០. ការត្រៀមបរិស្ថាន

ពិនិត្យកំណែ torch និងត្រៀមទិន្នន័យបញ្ចូលឧទាហរណ់សម្រាប់ជំពូកនេះ។

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))

In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

## ១. Attention សាមញ្ញបំផុត

គណនា attention score ដោយ dot product រវាង query និង input នីមួយៗ រួចធ្វើ normalization ដៃ។

In [ ]:
query = inputs[1]  # token បញ្ចូលទី ២ គឺជា query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)  # dot product (មិនចាំបាច់ transpose ទេ ព្រោះវាជា vector 1-dim)

print(attn_scores_2)

In [ ]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

## ២. Attention scores និង context vectors

គណនា scores សម្រាប់គ្រប់ token ក្នុងពេលតែមួយ និងទាញយក context vector តាមរយៈផលបូកថ្លឹង។

In [ ]:
query = inputs[1]  # token បញ្ចូលទី ២ គឺជា query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

In [ ]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

In [ ]:
print("Previous 2nd context vector:", context_vec_2)

## ៣. Trainable Q/K/V weights

ណែនាំម៉ាទ្រីសទម្ងន់ Query/Key/Value ដែលអាចបណ្តុះបណ្តាលបាន និងគណនា context vector ជាមួយវា។

In [ ]:
x_2 = inputs[1]  # ធាតុបញ្ចូលទី ២
d_in = inputs.shape[1]  # ទំហំ embedding បញ្ចូល, d=3
d_out = 2  # ទំហំ embedding លទ្ធផល, d=2

In [ ]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
query_2 = x_2 @ W_query  # _2 ព្រោះវាធៀបនឹងធាតុបញ្ចូលទី ២
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

In [ ]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

In [ ]:
keys_2 = keys[1]  # Python ចាប់ផ្តើមសន្ទស្សនពី ០ (0)
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

In [ ]:
attn_scores_2 = query_2 @ keys.T  # attention scores ទាំងអស់សម្រាប់ query ដែលបានផ្តល់
print(attn_scores_2)

In [ ]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

## ៤. ក្លាស SelfAttention_v2

បង្កប់ដំណើរការ attention ទាំងមូលចូលក្នុង nn.Module តែមួយ។

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

In [ ]:
# ប្រើម៉ាទ្រីសទម្ងន់ query និង key របស់វត្ថុ SelfAttention_v2 ពីផ្នែកមុនឡើងវិញ ដើម្បីភាពងាយស្រួល
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

## ៥. Causal mask

ប្រើ mask ត្រីកោណដើម្បីទប់មិនឱ្យ token មើលឃើញ token នៅខាងមុខវា (causal attention)។

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

## ៦. Dropout និង batch

បន្ថែម dropout លើ attention weights និងរៀបចំទិន្នន័យជា batch។

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6)  # បង្កើតម៉ាទ្រីសនៃលេខ ១

print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)  # បញ្ចូល ២ ដែលមាន ៦ tokens ក្នុងមួយៗ ហើយ token នីមួយៗមានវិមាត្រ embedding 3

## ៧. ក្លាស CausalAttention

ក្លាស attention ពេញលេញដែលរួមបញ្ចូល causal mask និង dropout។

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # ថ្មី
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))  # ថ្មី

    def forward(self, x):
        b, num_tokens, d_in = x.shape  # វិមាត្រ batch ថ្មី b
        # ចំពោះបញ្ចូលដែល `num_tokens` លើស `context_length` នេះនឹងបង្កកំហុសក្នុងការបង្កើត mask នៅខាងក្រោម។
        # ជាក់ស្តែង នេះមិនមែនជាបញ្ហាទេ ព្រោះ LLM (ជំពូក ៤-) ធានាថាបញ្ចូលមិនលើស `context_length` មុនពេលដល់ forward method នេះ។
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)  # បានប្តូរ transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)  # ថ្មី

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

## ៨. Multi-head attention តាម wrapper

បង្កើត multi-head attention ដោយរុំក្លាស CausalAttention ច្រើន head។

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1]  # នេះជាចំនួន tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

## ៩. ក្លាស MultiHeadAttention បង្រួម

ការអនុវត្ត multi-head attention ប្រកបដោយប្រសិទ្ធភាពក្នុងក្លាសតែមួយ។

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # បន្ថយវិមាត្រ projection ្យស្មើវិមាត្រលទ្ធផលដែលចង់បាន

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # ស្រទាប់ Linear សម្រាប់ផ្គុំលទ្ធផលរបស់ heads
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # ដូចក្នុង `CausalAttention` ចំពោះបញ្ចូលដែល `num_tokens` លើស `context_length` នឹងបង្កកំហុសក្នុងការបង្កើត mask នៅខាងក្រោម។
        # ជាក់ស្តែង នេះមិនមែនជាបញ្ហាទេ ព្រោះ LLM (ជំពូក ៤-) ធានាថាបញ្ចូលមិនលើស `context_length` មុនពេលដល់ forward method នេះ។

        keys = self.W_key(x)  # រូបរាង៖ (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # យើងបំបែកម៉ាទ្រីសដោយស្វ័យប្រវត្តិ ដោយបន្ថែមវិមាត្រ `num_heads`
        # ពន្លាវិមាត្រចុងក្រោយ៖ (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose៖ (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # គណនា scaled dot-product attention (ហៅម្យ៉ាងទៀតថា self-attention) ជាមួយ causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # dot product សម្រាប់ head នីមួយៗ

        # mask ដើមត្រូវបានកាត់តាមចំនួន tokens ហើយបំលែងទៅជា boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # ប្រើ mask ដើម្បីបំពេញ attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # រូបរាង៖ (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # ផ្គុំ heads ដែល self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # projection ជាជម្រើស

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

In [ ]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

In [ ]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ